In [0]:
import time
from functools import reduce
from operator import or_

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType


def _valor(c):
    """trim + string vazia vira NULL (ausencia uniforme na silver)"""
    return F.nullif(F.trim(c), F.lit(""))


t0 = time.time()

# ----------------------------------------------------------------
# 1. LEITURA UNICA da bronze
# ----------------------------------------------------------------
bronze = spark.read.table("vidaplus.bronze.fact_ibge_socioeconomico_bairros")

# ----------------------------------------------------------------
# 2. CHECKS DE CONTRATO (EDA 2026-11-13: bronze 100% limpa — rede de
#    seguranca executavel, conforme taxonomia do plano.md E1.8)
#    impossivel/contraditorio -> QUARENTENA:
#      - dominio fora do contrato do dicionario (idhm/IVS fora 0-1,
#        pct fora 0-100, populacao/area/densidade nao positivos,
#        renda/UBS negativos, ano fora 2021-2025)
#      - id_bairro fora do padrao <cod_municipio_ibge>-B<nn>
#        (padrao INFERIDO dos dados — dicionario nao explicita;
#        F2 usa exatamente os mesmos 144 ids)
#    NAO e contrato -> sem acao: densidade != populacao/area
#    (desvio max 0,29% na EDA; idem continuidade de estoque F4b)
# ----------------------------------------------------------------
INT_COLS = ["ano", "populacao_estimada", "num_unidades_basicas_saude"]
FLOAT_COLS = [
    "area_km2", "densidade_demografica_hab_km2", "renda_domiciliar_media_brl",
    "pct_domicilios_agua_encanada", "pct_domicilios_esgoto_sanitario",
    "pct_domicilios_coleta_lixo", "idhm", "pct_populacao_0_14",
    "pct_populacao_60_mais", "pct_populacao_dependente_sus",
    "cobertura_estrategia_saude_familia_pct", "indice_vulnerabilidade_social",
    "pct_domicilios_armazenamento_agua_inadequado",
]
COLS_TIPO = {**{c: IntegerType() for c in INT_COLS},
             **{c: DoubleType() for c in FLOAT_COLS}}

# Regra 1: valor presente que nao converte p/ o tipo do contrato
cast_invalido = reduce(
    or_,
    [_valor(c).isNotNull() & _valor(c).try_cast(t).isNull() for c, t in COLS_TIPO.items()],
)

# Regra 2: dominio fora do contrato do dicionario
_ano = _valor("ano").try_cast(IntegerType())
_pop = _valor("populacao_estimada").try_cast(IntegerType())
_ubs = _valor("num_unidades_basicas_saude").try_cast(IntegerType())
_area = _valor("area_km2").try_cast(DoubleType())
_dens = _valor("densidade_demografica_hab_km2").try_cast(DoubleType())
_renda = _valor("renda_domiciliar_media_brl").try_cast(DoubleType())
_idx01 = {c: _valor(c).try_cast(DoubleType()) for c in ("idhm", "indice_vulnerabilidade_social")}
_pct = {c: _valor(c).try_cast(DoubleType()) for c in FLOAT_COLS
        if c.startswith(("pct_", "cobertura_"))}

dominio_violado = (
    reduce(or_, [v.isNotNull() & ~v.between(0, 1) for v in _idx01.values()])
    | reduce(or_, [v.isNotNull() & ~v.between(0, 100) for v in _pct.values()])
    | (_pop.isNotNull() & (_pop <= 0))
    | (_area.isNotNull() & (_area <= 0))
    | (_dens.isNotNull() & (_dens <= 0))
    | (_renda.isNotNull() & (_renda < 0))
    | (_ubs.isNotNull() & (_ubs < 0))
    | (_ano.isNotNull() & ~_ano.between(2021, 2025))
)

# Regra 3: id_bairro fora do padrao inferido <cod_municipio_ibge>-B<nn>
_bairro_ok = F.trim("id_bairro").rlike(r"^\d{7}-B\d{2}$")
_prefixo_ok = F.substring(F.trim("id_bairro"), 1, 7) == F.trim("cod_municipio_ibge")
padrao_violado = F.coalesce(~_bairro_ok | ~_prefixo_ok, F.lit(False))

# Regra 0: chave obrigatoria do grao (bairro presente + ano converte p/ INT)
_bairro_chave = _valor("id_bairro")
chave_ausente = _bairro_chave.isNull() | _valor("ano").isNull() | _ano.isNull()

# Regra 4: grao duplicado (id_bairro x ano) -> TODAS as linhas do grao quarentenam
w_grao = Window.partitionBy(_bairro_chave, _ano)

checked = (
    bronze
    .withColumn("_chave_ausente", chave_ausente)
    .withColumn("_cnt_grao", F.count(F.lit(1)).over(w_grao))
    .withColumn("_cast_invalido", cast_invalido)
    .withColumn("_dominio_violado", dominio_violado)
    .withColumn("_padrao_violado", padrao_violado)
    .cache()
)

violou = F.coalesce(
    F.col("_chave_ausente") | (F.col("_cnt_grao") > 1) | F.col("_cast_invalido")
    | F.col("_dominio_violado") | F.col("_padrao_violado"),
    F.lit(False),
)

# ----------------------------------------------------------------
# 3. SILVER: espelho tipado (trim + try_cast); regras semanticas ficam p/ gold
# ----------------------------------------------------------------
silver = checked.filter(~violou).select(
    _valor("cod_municipio_ibge").alias("cod_municipio_ibge"),  # STRING: preserva zeros a esquerda (idem silver F2/F4)
    _valor("municipio").alias("municipio"),
    _valor("uf").alias("uf"),
    F.trim("id_bairro").alias("id_bairro"),  # nao-vazio garantido pela Regra 0
    _valor("nome_bairro").alias("nome_bairro"),
    _ano.alias("ano"),
    _pop.alias("populacao_estimada"),
    _area.alias("area_km2"),
    _dens.alias("densidade_demografica_hab_km2"),
    _renda.alias("renda_domiciliar_media_brl"),
    _pct["pct_domicilios_agua_encanada"].alias("pct_domicilios_agua_encanada"),
    _pct["pct_domicilios_esgoto_sanitario"].alias("pct_domicilios_esgoto_sanitario"),
    _pct["pct_domicilios_coleta_lixo"].alias("pct_domicilios_coleta_lixo"),
    _idx01["idhm"].alias("idhm"),
    _pct["pct_populacao_0_14"].alias("pct_populacao_0_14"),
    _pct["pct_populacao_60_mais"].alias("pct_populacao_60_mais"),
    _pct["pct_populacao_dependente_sus"].alias("pct_populacao_dependente_sus"),
    _ubs.alias("num_unidades_basicas_saude"),
    _pct["cobertura_estrategia_saude_familia_pct"].alias("cobertura_estrategia_saude_familia_pct"),
    _idx01["indice_vulnerabilidade_social"].alias("indice_vulnerabilidade_social"),
    _pct["pct_domicilios_armazenamento_agua_inadequado"].alias("pct_domicilios_armazenamento_agua_inadequado"),
    F.col("_arquivo_origem"),
    F.col("_ingerido_em"),
)

silver.write.mode("overwrite").saveAsTable("vidaplus.silver.fact_ibge_socioeconomico_bairros")

# ----------------------------------------------------------------
# 4. QUARENTENA: linhas que violaram regra (dado BRUTO, sem transformacao)
#    Precedencia da _regra_violada: chave > grao > cast > dominio > padrao
# ----------------------------------------------------------------
quarentena = (
    checked.filter(violou)
    .select(
        F.when(F.col("_chave_ausente"), F.lit("chave_obrigatoria_ausente"))
         .when(F.col("_cnt_grao") > 1, F.lit("grao_duplicado"))
         .when(F.col("_cast_invalido"), F.lit("cast_invalido"))
         .when(F.col("_dominio_violado"), F.lit("dominio_fora_do_contrato"))
         .when(F.col("_padrao_violado"), F.lit("id_bairro_fora_do_padrao"))
         .alias("_regra_violada"),
        F.col("_cnt_grao").alias("_total_no_grao"),
        *bronze.columns,
        F.current_timestamp().alias("_data_quarentena"),
    )
)

quarentena.write.mode("overwrite").saveAsTable("vidaplus.silver.quarentena_ibge_regras_violadas")

# ----------------------------------------------------------------
# 5. LOG da carga
# ----------------------------------------------------------------
n_bronze = bronze.count()
n_silver = silver.count()
_n_quar = quarentena.groupBy("_regra_violada").count().collect()
n_quarentena = quarentena.count()

print(f"Bronze:      {n_bronze:>8,} linhas")
print(f"Silver:      {n_silver:>8,} linhas | esperado 720")
print(f"Quarentena:  {n_quarentena:>8,} linhas | taxa: {n_quarentena / n_bronze:.2%}")
for r in _n_quar:
    print(f"   {r['_regra_violada']}: {r['count']}")
print(f"Tempo total: {time.time() - t0:.1f}s")

# Invariante de nao-perda: toda linha da bronze esta na silver OU na quarentena
assert n_silver + n_quarentena == n_bronze, (
    f"PERDA DE LINHAS: bronze={n_bronze}, silver+quarentena={n_silver + n_quarentena}"
)

checked.unpersist()  # libera o cache

In [0]:
# E1.7.8 — Comentarios UC na silver F5 (documentacao tecnica/de proveniencia;
# a gold recebe depois os comentarios de linguagem de negocio para o agente).
# Textos sem apóstrofo para evitar escape em SQL.

TB = "vidaplus.silver.fact_ibge_socioeconomico_bairros"

spark.sql(
    f"COMMENT ON TABLE {TB} IS '"
    "Silver - dados socioeconomicos IBGE por bairro x ano (720 linhas = 144 bairros x 5 anos, "
    "2021-2025). Espelho tipado da bronze (trim + try_cast); regras semanticas ficam para a gold. "
    "Grain: id_bairro + ano. Fonte: IBGE. EDA 2026-11-13: bronze 100% limpa — quarentena em "
    "vidaplus.silver.quarentena_ibge_regras_violadas e rede de seguranca executavel. Nota: Santa "
    "Maria (4318705/RS) nao e referenciada em F1/F4a — joins F1 x F5 nao cobrem esse municipio "
    "(orfao legitimo, nao e erro). Atencao gold: populacao_estimada e o denominador das taxas de "
    "incidencia.'"
)

COMMENTS = {
    "cod_municipio_ibge": "Municipio do bairro (STRING: preserva zeros a esquerda). FK -> dim_municipios (a carregar, E1.1). 12 municipios.",
    "municipio": "Nome do municipio (12 municipios em 4 UFs: SP, MG, BA, RS).",
    "uf": "UF do municipio: SP, MG, BA ou RS.",
    "id_bairro": "Chave do grao junto com ano. Formato <cod_municipio_ibge>-B<nn> (ex.: 3550308-B01). FK -> dim_bairros (144 valores; F2 usa os mesmos ids).",
    "nome_bairro": "Nome do bairro (estavel: cada id_bairro tem um unico nome em toda a serie).",
    "ano": "Ano de referencia do dado socioeconomico (2021-2025). Chave do grao junto com id_bairro.",
    "populacao_estimada": "Populacao residente estimada do bairro. DENOMINADOR das taxas de incidencia na gold.",
    "area_km2": "Area do bairro em km2 (> 0).",
    "densidade_demografica_hab_km2": "Habitantes por km2. Consistente com populacao/area (desvio max 0,29% na EDA) — nao e contrato, manter do fornecedor.",
    "renda_domiciliar_media_brl": "Renda domiciliar media mensal em reais correntes.",
    "pct_domicilios_agua_encanada": "% de domicilios com agua encanada (0-100).",
    "pct_domicilios_esgoto_sanitario": "% de domicilios com esgotamento sanitario (0-100).",
    "pct_domicilios_coleta_lixo": "% de domicilios com coleta regular de lixo (0-100).",
    "idhm": "Indice de Desenvolvimento Humano Municipal do bairro (0-1).",
    "pct_populacao_0_14": "% da populacao com 0-14 anos (0-100). Estrutura etaria.",
    "pct_populacao_60_mais": "% da populacao com 60+ anos (0-100). Estrutura etaria.",
    "pct_populacao_dependente_sus": "% da populacao sem plano de saude, dependente do SUS (0-100).",
    "num_unidades_basicas_saude": "Numero de UBS no bairro (INT >= 0).",
    "cobertura_estrategia_saude_familia_pct": "Cobertura da Estrategia Saude da Familia em % (0-100).",
    "indice_vulnerabilidade_social": "Indice de Vulnerabilidade Social do bairro (0-1); maior = mais vulneravel.",
    "pct_domicilios_armazenamento_agua_inadequado": "% de domicilios com armazenamento de agua inadequado (0-100) — proxy de criadouro de vetor (dengue).",
    "_arquivo_origem": "Metadado da bronze: arquivo CSV de origem da linha.",
    "_ingerido_em": "Metadado da bronze: timestamp de ingestao.",
}

for _col, _cm in COMMENTS.items():
    _cm = _cm.replace("\"", "'")  # aspas simples ja escapadas por construcao
    spark.sql(f"COMMENT ON COLUMN {TB}.{_col} IS '{_cm}'")

print(f"Comentarios aplicados: {len(COMMENTS)} colunas + tabela")

In [0]:
%sql
select * from vidaplus.silver.fact_ibge_socioeconomico_bairros;